# Azure 音频 Whisper（预览版）示例

> 注意：openai 库已有更新的版本。参见 https://github.com/openai/openai-python/discussions/742

本示例展示如何使用 Azure OpenAI Whisper 模型转录音频文件。

## 设置

首先，安装所需的依赖项。

In [ ]:
! pip install "openai>=0.28.1,<1.0.0"
! pip install python-dotenv

接下来，导入所需的库，并配置 Python OpenAI SDK 以使用 Azure OpenAI 服务。

> 注意：在本示例中，我们通过在代码中设置变量来让库使用 Azure API。在开发时，建议改为设置环境变量：

```
OPENAI_API_BASE
OPENAI_API_KEY
OPENAI_API_TYPE
OPENAI_API_VERSION
```

In [1]:
import os
import dotenv
import openai


dotenv.load_dotenv()

True


要正确访问 Azure OpenAI 服务，我们需要在 [Azure 门户](https://portal.azure.com) 中创建相应的资源（关于具体操作，可参阅 [Microsoft 文档](https://learn.microsoft.com/en-us/azure/cognitive-services/openai/how-to/create-resource?pivots=web-portal) 中的详细指南）

资源创建完成后，首先需要用到的是它的终结点（endpoint）。你可以在 *"资源管理"* 部分下的 *"密钥和终结点"* 中找到终结点。拿到它之后，我们用这些信息来设置 SDK：

In [2]:
openai.api_base = os.environ["OPENAI_API_BASE"]

# Min API version that supports Whisper
openai.api_version = "2023-09-01-preview"

# Enter the deployment_id to use for the Whisper model
deployment_id = "<deployment-id-for-your-whisper-model>"

### 身份验证

Azure OpenAI 服务支持多种身份验证机制，包括 API 密钥和 Azure 凭据。

In [3]:
# set to True if using Azure Active Directory authentication
use_azure_active_directory = False


#### 使用 API 密钥进行身份验证

要将 OpenAI SDK 设置为使用 *Azure API 密钥*，需要将 `api_type` 设置为 `azure`，并将 `api_key` 设置为与你的终结点关联的密钥（你可以在 [Azure 门户](https://portal.azure.com) 的 *"资源管理"* 下的 *"密钥和终结点"* 中找到该密钥）

In [4]:
if not use_azure_active_directory:
    openai.api_type = 'azure'
    openai.api_key = os.environ["OPENAI_API_KEY"]

#### 使用 Azure Active Directory 进行身份验证
现在来看看如何通过 Microsoft Active Directory 身份验证获取密钥。

In [ ]:
from azure.identity import DefaultAzureCredential

if use_azure_active_directory:
    default_credential = DefaultAzureCredential()
    token = default_credential.get_token("https://cognitiveservices.azure.com/.default")

    openai.api_type = 'azure_ad'
    openai.api_key = token.token

令牌在一段时间内有效，过期后即失效。为确保每个请求都携带有效的令牌，你可以通过挂接 requests.auth 来刷新即将过期的令牌：

In [ ]:
import typing
import time
import requests

if typing.TYPE_CHECKING:
    from azure.core.credentials import TokenCredential

class TokenRefresh(requests.auth.AuthBase):

    def __init__(self, credential: "TokenCredential", scopes: typing.List[str]) -> None:
        self.credential = credential
        self.scopes = scopes
        self.cached_token: typing.Optional[str] = None

    def __call__(self, req):
        if not self.cached_token or self.cached_token.expires_on - time.time() < 300:
            self.cached_token = self.credential.get_token(*self.scopes)
        req.headers["Authorization"] = f"Bearer {self.cached_token.token}"
        return req

if use_azure_active_directory:
    session = requests.Session()
    session.auth = TokenRefresh(default_credential, ["https://cognitiveservices.azure.com/.default"])

    openai.requestssession = session

## 音频转录

音频转录，即语音转文本，是将口语转换为文字的过程。使用 `openai.Audio.transcribe` 方法将音频文件流转录为文本。

你可以从 [GitHub 上的 Azure AI Speech SDK 仓库](https://github.com/Azure-Samples/cognitive-services-speech-sdk/tree/master/sampledata/audiofiles) 获取示例音频文件。

In [9]:
# download sample audio file
import requests

sample_audio_url = "https://github.com/Azure-Samples/cognitive-services-speech-sdk/raw/master/sampledata/audiofiles/wikipediaOcelot.wav"
audio_file = requests.get(sample_audio_url)
with open("wikipediaOcelot.wav", "wb") as f:
    f.write(audio_file.content)

In [ ]:
transcription = openai.Audio.transcribe(
    file=open("wikipediaOcelot.wav", "rb"),
    model="whisper-1",
    deployment_id=deployment_id,
)
print(transcription.text)